# Week 1 pilot: does the phone channel break a frozen SSL-AASIST?

This notebook answers the pilot's first question for the main detector. **When a correctly detected utterance goes through a phone-call stage, does the detector's ranking break (EER rises), or only its threshold (real callers get rejected at the threshold set on clean audio)?**

**What it does**
1. Loads Tak et al.'s released SSL-AASIST checkpoint (XLS-R + AASIST, trained on ASVspoof 2019 LA). The fairseq front-end is replaced by an identical Hugging Face model, so **no fairseq install is needed**. Loading is strict, and the clean EER is checked against the published figure.
2. Takes a seeded, stratified subset of ASVspoof 2019 LA (eval: 1,000 bona fide + 1,000 spoof, all 13 attacks; dev: 500 + 500 for the threshold).
3. Passes every eval utterance through each call stage, sample-aligned with its clean original: loudness normalisation, telephone band-pass, 8 kHz resampling, G.711, G.711 with 10% frame loss, GSM-FR, Opus 8 kbps, and AMR-NB if your ffmpeg can encode it.
4. Scores everything and reports, per stage: EER (ranking), real callers rejected and fakes accepted at the threshold transferred from clean dev audio (operating point), min/act DCF at two priors, the per-class score shift with 95% CIs, and the pre-registered pilot decision.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Everything that matters is saved to `MyDrive/fyp_pilot`, so a disconnected session resumes where it stopped.

**Time:** about 15–25 min to download data the first time, then about 15–25 min of scoring on a T4.

In [ ]:
#@title 1 · Settings and Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, pathlib
CFG = dict(
    SEED=1234,
    N_EVAL_BONA=1000, N_EVAL_SPOOF=1000,   # eval subset (spoof split evenly over A07-A19)
    N_DEV_BONA=500,   N_DEV_SPOOF=500,     # dev subset, used only to set the clean threshold
    BATCH=16,
    OUT='/content/drive/MyDrive/fyp_pilot',
    WORK='/content/work',
    # ASVspoof 2019 LA (Edinburgh DataShare). If the download fails, download LA.zip in a browser,
    # put it in your Drive and set LA_ZIP to its path, e.g. '/content/drive/MyDrive/LA.zip'.
    LA_ZIP_URL='https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip',
    LA_ZIP='',
    # Tak et al.'s released checkpoints (Google Drive folder from the repository README).
    # If gdown fails, download the LA model manually and set CKPT to its Drive path.
    CKPT_FOLDER='https://drive.google.com/drive/folders/1c4ywztEVlYVijfwbGLl9OEa1SNtFKppB',
    CKPT='',
)
for d in (CFG['OUT'], CFG['WORK'], f"{CFG['OUT']}/scores", f"{CFG['OUT']}/figures"):
    os.makedirs(d, exist_ok=True)
print(json.dumps(CFG, indent=1))

In [ ]:
#@title 2 · Install packages and check codecs
!pip -q install transformers soundfile gdown pandas matplotlib
import subprocess, re, shutil
def encoders(ff='ffmpeg'):
    out = subprocess.run([ff, '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
    return {n for n in ('libgsm', 'libopus', 'pcm_mulaw', 'libopencore_amrnb') if re.search(rf'\s{n}\s', out)}
FFMPEG = 'ffmpeg'
enc = encoders()
if 'libopencore_amrnb' not in enc:
    # Colab's ffmpeg usually lacks the AMR-NB encoder; try a static build that includes it.
    try:
        !wget -q -O /content/ffs.tar.xz https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz
        !mkdir -p /content/ffs && tar -xf /content/ffs.tar.xz -C /content/ffs --strip-components=1
        if 'libopencore_amrnb' in encoders('/content/ffs/ffmpeg'):
            FFMPEG, enc = '/content/ffs/ffmpeg', encoders('/content/ffs/ffmpeg')
    except Exception as e:
        print('static ffmpeg not available:', e)
print('ffmpeg:', FFMPEG)
print('encoders found:', sorted(enc))
if 'libopencore_amrnb' not in enc:
    print('AMR-NB encoder not available: AMR conditions will be skipped (GSM-FR covers mobile).')

In [ ]:
#@title 3 · Get Tak et al.'s code and the released LA checkpoint
import glob
REPO = '/content/SSL_Anti-spoofing'
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/TakHemlata/SSL_Anti-spoofing {REPO}
REPO_COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('repo commit:', REPO_COMMIT)

ck_dir = f"{CFG['OUT']}/checkpoints"
CKPT = CFG['CKPT']
if not CKPT:
    os.makedirs(ck_dir, exist_ok=True)
    found = [p for p in glob.glob(f'{ck_dir}/**/*.pth', recursive=True) if 'LA' in os.path.basename(p)]
    if not found:
        !gdown -q --folder {CFG['CKPT_FOLDER']} -O {ck_dir}
        found = [p for p in glob.glob(f'{ck_dir}/**/*.pth', recursive=True) if 'LA' in os.path.basename(p)]
    assert found, ('Could not download the checkpoint. Open the Google Drive folder in CFG["CKPT_FOLDER"], '
                   'download the LA model, put it in your Drive, set CFG["CKPT"] to its path and rerun.')
    CKPT = sorted(found)[0]
print('checkpoint:', CKPT, f'({os.path.getsize(CKPT)/1e6:.0f} MB)')

In [ ]:
#@title 4 · Get ASVspoof 2019 LA and extract a seeded subset to Drive
import zipfile, numpy as np, pandas as pd
SUB = f"{CFG['OUT']}/subset"
subset_csv = f"{CFG['OUT']}/subset.csv"
if not os.path.exists(subset_csv):
    zpath = CFG['LA_ZIP'] or f"{CFG['WORK']}/LA.zip"
    if not os.path.exists(zpath):
        !wget -q --show-progress -O {zpath} "{CFG['LA_ZIP_URL']}"
    z = zipfile.ZipFile(zpath)
    names = z.namelist()
    def member(suffix):
        m = [n for n in names if n.endswith(suffix)]
        assert m, suffix; return m[0]
    def protocol(part):
        rows = z.read(member(f'ASVspoof2019.LA.cm.{part}.trl.txt')).decode().split('\n')
        df = pd.DataFrame([r.split() for r in rows if r.strip()], columns=['speaker', 'utt', 'sys', 'attack', 'key'])
        df['label'] = (df.key == 'bonafide').astype(int); df['part'] = part
        return df
    rng = np.random.default_rng(CFG['SEED'])
    def pick(df, n_bona, n_spoof):
        bona = df[df.label == 1].sample(n_bona, random_state=CFG['SEED'])
        sp = df[df.label == 0]; atks = sorted(sp.attack.unique()); per = n_spoof // len(atks)
        spoof = pd.concat([sp[sp.attack == a].sample(per, random_state=CFG['SEED']) for a in atks])
        return pd.concat([bona, spoof])
    ev, dv = protocol('eval'), protocol('dev')
    subset = pd.concat([pick(ev, CFG['N_EVAL_BONA'], CFG['N_EVAL_SPOOF']), pick(dv, CFG['N_DEV_BONA'], CFG['N_DEV_SPOOF'])])
    os.makedirs(SUB, exist_ok=True)
    for _, r in subset.iterrows():
        folder = 'ASVspoof2019_LA_eval' if r.part == 'eval' else 'ASVspoof2019_LA_dev'
        with open(f'{SUB}/{r.utt}.flac', 'wb') as f:
            f.write(z.read(member(f'{folder}/flac/{r.utt}.flac')))
    subset.to_csv(subset_csv, index=False)
subset = pd.read_csv(subset_csv)
subset['path'] = SUB + '/' + subset.utt + '.flac'
print(subset.groupby(['part', 'label']).size().rename('n'))
print('eval attacks:', sorted(subset[(subset.part == 'eval') & (subset.label == 0)].attack.unique()))

In [ ]:
%%writefile /content/pilot_lib.py
# ---------------------------------------------------------------------------
# Pilot library: call-stage simulator, alignment, SSL-AASIST loader, metrics.
# Kept in one file so it can be tested outside Colab and pasted into the notebook.
# ---------------------------------------------------------------------------
import os, re, sys, types, json, zlib, shutil, tempfile, subprocess
import numpy as np
import soundfile as sf
from scipy.signal import butter, sosfiltfilt, correlate

SR = 16000
CUT = 64600          # SSL-AASIST input length (~4.04 s), as in Tak et al.'s code


# ----------------------------- audio helpers ------------------------------
def tak_pad(x, max_len=CUT):
    """Exactly Tak et al.'s pad(): truncate, or tile short clips."""
    if len(x) >= max_len:
        return x[:max_len]
    reps = int(max_len / len(x)) + 1
    return np.tile(x, reps)[:max_len]


def _run(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + '\n' + r.stderr.decode(errors='ignore')[-800:])


def ffmpeg_roundtrip(x, enc_args, codec_sr=8000, ext='wav', ffmpeg='ffmpeg'):
    """16 kHz float -> (resample to codec_sr) -> encode -> decode -> 16 kHz float."""
    with tempfile.TemporaryDirectory() as d:
        inp, mid, out = f'{d}/in.wav', f'{d}/mid.{ext}', f'{d}/out.wav'
        sf.write(inp, np.clip(x, -1, 1), SR, subtype='PCM_16')
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', inp, '-ac', '1', '-ar', str(codec_sr)] + enc_args + [mid])
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', mid, '-ac', '1', '-ar', str(SR), '-c:a', 'pcm_f32le', out])
        y, _ = sf.read(out, dtype='float32')
    return y


def align_to(ref, y, max_lag=1600):
    """Remove codec delay: find the lag that best matches ref, shift y, trim/pad to len(ref)."""
    n = min(len(ref), len(y), 2 * SR)
    a, b = ref[:n].astype(np.float64), y[:n].astype(np.float64)
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        lag = 0
    else:
        c = correlate(b, a, mode='full', method='fft')
        lags = np.arange(-n + 1, n)
        keep = np.abs(lags) <= max_lag
        lag = int(lags[keep][np.argmax(c[keep])])
    if lag > 0:
        y = y[lag:]
    elif lag < 0:
        y = np.concatenate([np.zeros(-lag, dtype=y.dtype), y])
    if len(y) < len(ref):
        y = np.concatenate([y, np.zeros(len(ref) - len(y), dtype=y.dtype)])
    return y[:len(ref)].astype(np.float32), lag


def bandpass(x, lo=300, hi=3400, order=6):
    sos = butter(order, [lo, hi], btype='bandpass', fs=SR, output='sos')
    return sosfiltfilt(sos, x).astype(np.float32)   # zero-phase: no delay


def rms_normalise(x, target_dbfs=-26.0):
    rms = np.sqrt(np.mean(x ** 2)) + 1e-12
    y = x * (10 ** (target_dbfs / 20) / rms)
    peak = np.max(np.abs(y))
    return (y / peak * 0.99 if peak > 0.99 else y).astype(np.float32)


def frame_loss(x, utt_id, rate, seed, frame=320, fade=40):
    """Drop 20 ms frames at random (seeded per utterance) and conceal by repeating
    the previous frame, attenuated, with short cross-fades. A simplified packet-loss
    concealment, documented as such; not the G.711 Appendix I algorithm."""
    rng = np.random.default_rng((zlib.crc32(utt_id.encode()) ^ seed) & 0xFFFFFFFF)
    y = x.copy()
    n = len(x) // frame
    lost = rng.random(n) < rate
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    run = 0
    for i in range(1, n):
        s = i * frame
        if lost[i]:
            run += 1
            seg = y[s - frame:s] * (0.7 ** run)          # repeat the last frame, attenuated
            seg[:fade] = y[s - 1] * (1 - ramp) + seg[:fade] * ramp   # smooth join, no click
            y[s:s + frame] = seg
        else:
            if run:   # fade back in from the concealed frame
                y[s:s + fade] = y[s - fade:s] * (1 - ramp) + x[s:s + fade] * ramp
            run = 0
    return y, float(lost.mean())


# ----------------------------- conditions ---------------------------------
def available_encoders(ffmpeg='ffmpeg'):
    out = subprocess.run([ffmpeg, '-hide_banner', '-encoders'], stdout=subprocess.PIPE, stderr=subprocess.PIPE).stdout.decode()
    return {name for name in ('libgsm', 'libopus', 'pcm_mulaw', 'libopencore_amrnb') if re.search(rf'\s{name}\s', out)}


def make_conditions(encoders, seed, ffmpeg='ffmpeg'):
    """Returns {name: (description, fn(x, utt_id) -> (y, info))}. Every output is
    16 kHz, the same length as the input and sample-aligned with it."""
    C = {}
    C['clean'] = ('No processing', lambda x, u: (x, {}))
    C['clean_rms'] = ('Clean, loudness-normalised to -26 dBFS (loudness/AGC control)',
                      lambda x, u: (rms_normalise(x), {}))
    C['bandpass_300_3400'] = ('Telephone band 300-3,400 Hz, zero-phase filter (band-limit control)',
                              lambda x, u: (bandpass(x), {}))

    def coded(args, ext, sr=8000):
        def fn(x, u):
            y, lag = align_to(x, ffmpeg_roundtrip(x, args, sr, ext, ffmpeg))
            return y, {'lag': lag}
        return fn

    C['nb_resample'] = ('16 -> 8 -> 16 kHz resampling only (control for all 8 kHz codecs)',
                        coded(['-c:a', 'pcm_s16le'], 'wav'))
    if 'pcm_mulaw' in encoders:
        C['g711u'] = ('G.711 mu-law, 64 kbps', coded(['-c:a', 'pcm_mulaw'], 'wav'))

        def g711_loss(x, u):
            y, info = C['g711u'][1](x, u)
            y, lost = frame_loss(y, u, 0.10, seed)
            return y, dict(info, lost=lost)
        C['g711u_loss10'] = ('G.711 mu-law with 10% 20 ms frame loss, repetition concealment', g711_loss)
    if 'libgsm' in encoders:
        C['gsm_fr'] = ('GSM full-rate, 13 kbps (2G mobile)', coded(['-c:a', 'libgsm'], 'gsm'))
    if 'libopencore_amrnb' in encoders:
        C['amrnb_12k2'] = ('AMR-NB 12.2 kbps (mobile)', coded(['-c:a', 'libopencore_amrnb', '-b:a', '12.2k'], 'amr'))
        C['amrnb_4k75'] = ('AMR-NB 4.75 kbps (mobile, low rate)', coded(['-c:a', 'libopencore_amrnb', '-b:a', '4.75k'], 'amr'))
    if 'libopus' in encoders:
        C['opus_nb_8k'] = ('Opus narrowband, 8 kbps, VoIP mode',
                           coded(['-c:a', 'libopus', '-b:a', '8k', '-application', 'voip'], 'opus'))
    return C


# ----------------------------- metrics ------------------------------------
def det_curve(tar, non):
    """Identical to Tak et al.'s compute_det_curve (target = bona fide)."""
    n = tar.size + non.size
    s = np.concatenate([tar, non]); l = np.concatenate([np.ones(tar.size), np.zeros(non.size)])
    idx = np.argsort(s, kind='mergesort'); l = l[idx]
    tsum = np.cumsum(l); nsum = non.size - (np.arange(1, n + 1) - tsum)
    frr = np.concatenate([[0], tsum / tar.size]); far = np.concatenate([[1], nsum / non.size])
    thr = np.concatenate([[s[idx[0]] - 0.001], s[idx]])
    return frr, far, thr


def eer(tar, non):
    frr, far, thr = det_curve(tar, non)
    i = np.argmin(np.abs(frr - far))
    return float(np.mean([frr[i], far[i]])), float(thr[i])


def rates_at(tar, non, t):
    """Bona fide rejected (false alarm on a real caller) and spoof accepted (miss), at threshold t.
    A score >= t is accepted as bona fide."""
    return float(np.mean(tar < t)), float(np.mean(non >= t))


def dcf(p_fa_bona, p_miss_spoof, p_spoof, c_rej_bona=1.0, c_acc_spoof=10.0):
    """Normalised detection cost. 'Rejecting bona fide' and 'accepting spoof' are the two errors.
    Defaults follow ASVspoof 5 (C=1 and 10, P_spoof=0.05)."""
    cost = c_rej_bona * (1 - p_spoof) * p_fa_bona + c_acc_spoof * p_spoof * p_miss_spoof
    return float(cost / min(c_rej_bona * (1 - p_spoof), c_acc_spoof * p_spoof))


def min_dcf(tar, non, p_spoof, c1=1.0, c2=10.0):
    frr, far, _ = det_curve(tar, non)
    return float(min(dcf(fr, fa, p_spoof, c1, c2) for fr, fa in zip(frr, far)))


def boot_ci(d, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    m = [rng.choice(d, d.size, replace=True).mean() for _ in range(n)]
    return float(np.mean(d)), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


PRIORS = {'asv5': (0.05, 1.0, 10.0), 'balanced': (0.5, 1.0, 1.0)}


def summarise(scores, clean, thr):
    """scores/clean: DataFrames indexed by utt with columns score, label(1=bona fide), correct_clean."""
    out = {}
    for subset, mask in (('all', np.ones(len(scores), bool)), ('clean_correct', scores['correct_clean'].values)):
        s = scores[mask]; c = clean.loc[s.index]
        tar, non = s[s.label == 1].score.values, s[s.label == 0].score.values
        if tar.size < 2 or non.size < 2:          # e.g. an empty clean-correct subset
            out[subset] = {'eer': np.nan, 'fa_bona_at_thr': np.nan, 'miss_spoof_at_thr': np.nan,
                           'n_bona': int(tar.size), 'n_spoof': int(non.size)}
            continue
        e, _ = eer(tar, non)
        fa, miss = rates_at(tar, non, thr)
        r = {'eer': e, 'fa_bona_at_thr': fa, 'miss_spoof_at_thr': miss, 'hter_at_thr': (fa + miss) / 2}
        for k, (p, c1, c2) in PRIORS.items():
            r[f'minDCF_{k}'] = min_dcf(tar, non, p, c1, c2)
            r[f'actDCF_{k}'] = dcf(fa, miss, p, c1, c2)
            r[f'calib_loss_{k}'] = r[f'actDCF_{k}'] - r[f'minDCF_{k}']
        for lab, name in ((1, 'bona'), (0, 'spoof')):
            d = (s[s.label == lab].score.values - c[c.label == lab].score.values)
            m, lo, hi = boot_ci(d)
            r[f'shift_{name}_mean'], r[f'shift_{name}_lo'], r[f'shift_{name}_hi'] = m, lo, hi
        r['n_bona'], r['n_spoof'] = int(tar.size), int(non.size)
        out[subset] = r
    return out


def decision(row_clean, row):
    """Pre-registered pilot rule: clean EER <= 5% AND (EER rises >= 10 points OR
    real callers rejected at the transferred threshold rise >= 15 points)."""
    ok_clean = row_clean['eer'] <= 0.05
    d_eer = row['eer'] - row_clean['eer']
    d_fa = row['fa_bona_at_thr'] - row_clean['fa_bona_at_thr']
    counts = ok_clean and (d_eer >= 0.10 or d_fa >= 0.15)
    if not counts:
        kind = 'no_effect'
    elif d_eer >= 0.10:
        kind = 'ranking_breaks'
    else:
        kind = 'threshold_only'
    return counts, kind, d_eer, d_fa


# ----------------------------- model loader -------------------------------
def xlsr_config(n_layers=24, conv_bias=True):
    from transformers import Wav2Vec2Config
    return Wav2Vec2Config(
        hidden_size=1024, num_hidden_layers=n_layers, num_attention_heads=16, intermediate_size=4096,
        hidden_act='gelu', hidden_dropout=0.0, activation_dropout=0.0, attention_dropout=0.0,
        feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0, layer_norm_eps=1e-5,
        feat_extract_norm='layer', feat_extract_activation='gelu', conv_dim=(512,) * 7,
        conv_stride=(5, 2, 2, 2, 2, 2, 2), conv_kernel=(10, 3, 3, 3, 3, 2, 2), conv_bias=conv_bias,
        num_conv_pos_embeddings=128, num_conv_pos_embedding_groups=16, do_stable_layer_norm=True,
        apply_spec_augment=False, mask_time_prob=0.0, mask_feature_prob=0.0)


IGNORED_FS_PREFIXES = ('quantizer.', 'project_q.', 'final_proj.', 'project_inp.', 'target_glu.', 'input_quantizer.', 'mask_emb')


def map_fairseq_to_hf(sd_fs, hf_keys):
    """Rename fairseq wav2vec 2.0 parameter names to Hugging Face Wav2Vec2Model names.
    Module names checked against the fairseq commit bundled in Tak et al.'s repository."""
    hf_keys = set(hf_keys)
    param_wn = any('parametrizations.weight.original0' in k for k in hf_keys)
    out, unused = {}, []
    for k, v in sd_fs.items():
        nk = None
        if m := re.fullmatch(r'feature_extractor\.conv_layers\.(\d+)\.0\.(weight|bias)', k):
            nk = f'feature_extractor.conv_layers.{m[1]}.conv.{m[2]}'
        elif m := re.fullmatch(r'feature_extractor\.conv_layers\.(\d+)\.2\.1\.(weight|bias)', k):
            nk = f'feature_extractor.conv_layers.{m[1]}.layer_norm.{m[2]}'
        elif k in ('layer_norm.weight', 'layer_norm.bias'):
            nk = 'feature_projection.' + k
        elif m := re.fullmatch(r'post_extract_proj\.(weight|bias)', k):
            nk = f'feature_projection.projection.{m[1]}'
        elif k == 'encoder.pos_conv.0.bias':
            nk = 'encoder.pos_conv_embed.conv.bias'
        elif k == 'encoder.pos_conv.0.weight_g':
            nk = 'encoder.pos_conv_embed.conv.' + ('parametrizations.weight.original0' if param_wn else 'weight_g')
        elif k == 'encoder.pos_conv.0.weight_v':
            nk = 'encoder.pos_conv_embed.conv.' + ('parametrizations.weight.original1' if param_wn else 'weight_v')
        elif m := re.fullmatch(r'encoder\.layers\.(\d+)\.(.+)', k):
            rest = (m[2].replace('self_attn_layer_norm', 'layer_norm').replace('self_attn.', 'attention.')
                    .replace('fc1.', 'feed_forward.intermediate_dense.').replace('fc2.', 'feed_forward.output_dense.'))
            nk = f'encoder.layers.{m[1]}.{rest}'
        elif k in ('encoder.layer_norm.weight', 'encoder.layer_norm.bias'):
            nk = k
        elif k == 'mask_emb':
            nk = 'masked_spec_embed'
        if nk is None or nk not in hf_keys:
            unused.append(k)
        else:
            out[nk] = v
    missing = sorted(hf_keys - set(out))
    return out, unused, missing


def build_ssl_aasist(repo_dir, ckpt_path, device='cpu'):
    """Tak et al.'s SSL-AASIST with the fairseq front-end replaced by an identical
    Hugging Face Wav2Vec2Model, so no fairseq install is needed. Loads strictly."""
    import torch, torch.nn as nn
    from transformers import Wav2Vec2Model
    sd = torch.load(ckpt_path, map_location='cpu')
    if isinstance(sd, dict) and 'state_dict' in sd and isinstance(sd['state_dict'], dict):
        sd = sd['state_dict']
    sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
    pre = 'ssl_model.model.'
    fs = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
    n_layers = 1 + max(int(m[1]) for k in fs if (m := re.match(r'encoder\.layers\.(\d+)\.', k)))
    conv_bias = 'feature_extractor.conv_layers.0.0.bias' in fs
    cfg = xlsr_config(n_layers, conv_bias)

    class HFSSL(nn.Module):
        def __init__(self, device):
            super().__init__()
            self.model = Wav2Vec2Model(cfg)
            self.out_dim = cfg.hidden_size
            self.device = device

        def extract_feat(self, x, output_hidden_states=False):
            if x.ndim == 3:
                x = x[:, :, 0]
            o = self.model(x, output_hidden_states=output_hidden_states)
            if output_hidden_states:
                self.last_hidden_states = o.hidden_states
            return o.last_hidden_state

    if 'fairseq' not in sys.modules:                 # model.py imports fairseq at the top; never used here
        sys.modules['fairseq'] = types.ModuleType('fairseq')
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    import model as tak
    tak.SSLModel = HFSSL
    net = tak.Model(None, device)
    hf_keys = list(net.ssl_model.model.state_dict().keys())
    mapped, unused, missing = map_fairseq_to_hf(fs, hf_keys)
    bad_unused = [k for k in unused if not k.startswith(IGNORED_FS_PREFIXES)]
    missing = [k for k in missing if k != 'masked_spec_embed']
    if bad_unused or missing:
        raise RuntimeError(f'Weight mapping failed.\nUnmapped checkpoint keys: {bad_unused[:20]}\nMissing model keys: {missing[:20]}')
    full = {k: v for k, v in sd.items() if not k.startswith(pre)}
    full.update({pre + k: v for k, v in mapped.items()})
    if 'masked_spec_embed' in hf_keys and 'masked_spec_embed' not in mapped:   # only exists in some versions
        full[pre + 'masked_spec_embed'] = net.ssl_model.model.masked_spec_embed.detach()
    net.load_state_dict(full, strict=True)
    report = {'xlsr_layers': n_layers, 'conv_bias': conv_bias, 'ssl_params_loaded': len(mapped),
              'ignored_pretraining_params': len(unused), 'backend_params_loaded': len(sd) - len(fs)}
    return net.to(device).eval(), report

In [ ]:
#@title 6 · Load SSL-AASIST (strict) and run a quick sanity check
import sys, torch, numpy as np, soundfile as sf
sys.path.insert(0, '/content'); import importlib, pilot_lib as P; importlib.reload(P)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'Switch the runtime to a T4 GPU first.'
model, load_report = P.build_ssl_aasist(REPO, CKPT, dev)
print('loaded:', load_report)
with torch.no_grad():
    s = subset[subset.part == 'dev'].groupby('label').head(8)
    x = torch.tensor(np.stack([P.tak_pad(sf.read(p, dtype='float32')[0]) for p in s.path])).to(dev)
    sc = model(x)[:, 1].cpu().numpy()
print('bona fide scores:', np.round(sc[s.label.values == 1], 2))
print('spoof scores    :', np.round(sc[s.label.values == 0], 2))
print('(bona fide should be clearly higher than spoof)')

In [ ]:
#@title 7 · Score dev (clean) and eval (every call stage) — resumes if interrupted
import pandas as pd, time
from concurrent.futures import ThreadPoolExecutor
CONDS = P.make_conditions(enc, CFG['SEED'], FFMPEG)
print('conditions:', list(CONDS))
SC = f"{CFG['OUT']}/scores"

def prep(row, cond):
    x, sr = sf.read(row.path, dtype='float32'); assert sr == P.SR
    y, info = CONDS[cond][1](x, row.utt)
    return P.tak_pad(y), info

@torch.no_grad()
def score_set(df, cond):
    out = f'{SC}/{df.part.iloc[0]}__{cond}.csv'
    if os.path.exists(out):
        return pd.read_csv(out)
    t0, rows = time.time(), []
    with ThreadPoolExecutor(4) as ex:
        items = list(df.itertuples())
        for i in range(0, len(items), CFG['BATCH']):
            chunk = items[i:i + CFG['BATCH']]
            res = list(ex.map(lambda r: prep(r, cond), chunk))
            x = torch.tensor(np.stack([r[0] for r in res])).to(dev)
            s = model(x)[:, 1].float().cpu().numpy()
            for r, (_, info), v in zip(chunk, res, s):
                rows.append(dict(utt=r.utt, speaker=r.speaker, attack=r.attack, label=r.label, cond=cond,
                                 score=float(v), lag=info.get('lag', 0), lost=info.get('lost', 0.0)))
    d = pd.DataFrame(rows); d.to_csv(out, index=False)
    print(f'{df.part.iloc[0]:4s} {cond:18s} {len(d)} utts in {time.time()-t0:.0f}s')
    return d

dev_clean = score_set(subset[subset.part == 'dev'], 'clean')
EV = subset[subset.part == 'eval']
scores = {c: score_set(EV, c) for c in CONDS}

In [ ]:
#@title 8 · Reproduction check against the published result
eer_dev, THR = P.eer(dev_clean[dev_clean.label == 1].score.values, dev_clean[dev_clean.label == 0].score.values)
c = scores['clean']
eer_eval, _ = P.eer(c[c.label == 1].score.values, c[c.label == 0].score.values)
print(f'dev clean EER  {100*eer_dev:.2f}%  -> threshold {THR:.3f} (transferred to every condition)')
print(f'eval clean EER {100*eer_eval:.2f}%  (published for this checkpoint on full 2019 LA eval: about 0.2%; '
      f'2021 LA: 0.82%)')
REPRODUCED = eer_eval < 0.01
print('checkpoint:', os.path.basename(CKPT))
print('REPRODUCED' if REPRODUCED else 'NOT REPRODUCED: check the load report and audio before trusting any result below.')

In [ ]:
#@title 9 · Results table and the pilot decision
clean = scores['clean'].set_index('utt')
clean['correct_clean'] = (clean.score >= THR) == (clean.label == 1)
rows, base = [], None
for cname, d in scores.items():
    d = d.set_index('utt').loc[clean.index]
    d['correct_clean'] = clean.correct_clean
    summ = P.summarise(d, clean, THR)
    for subset_name, r in summ.items():
        rows.append(dict(detector='SSL-AASIST', condition=cname, subset=subset_name, **r))
res = pd.DataFrame(rows)
cl = res[res.condition == 'clean'].set_index('subset')
dec = []
for _, r in res.iterrows():
    counts, kind, d_eer, d_fa = P.decision(cl.loc[r.subset], r)
    dec.append((counts, kind, d_eer, d_fa))
res[['counts', 'kind', 'delta_eer', 'delta_fa_bona']] = pd.DataFrame(dec, index=res.index)
res.to_csv(f"{CFG['OUT']}/pilot_results.csv", index=False)
show = res[res.subset == 'all'][['condition', 'eer', 'fa_bona_at_thr', 'miss_spoof_at_thr', 'hter_at_thr',
                                 'minDCF_asv5', 'actDCF_asv5', 'shift_bona_mean', 'shift_spoof_mean',
                                 'delta_eer', 'delta_fa_bona', 'counts', 'kind']]
pct = ['eer', 'fa_bona_at_thr', 'miss_spoof_at_thr', 'hter_at_thr', 'delta_eer', 'delta_fa_bona']
display(show.style.format({**{k: '{:.1%}' for k in pct}, 'minDCF_asv5': '{:.3f}', 'actDCF_asv5': '{:.3f}',
                           'shift_bona_mean': '{:+.2f}', 'shift_spoof_mean': '{:+.2f}'}))
print('Decision rule: clean EER <= 5% AND (EER +10 points OR real callers rejected +15 points).')
for k, g in show.groupby('kind'):
    print(f'{k:15s}: {", ".join(g.condition)}')

In [ ]:
#@title 10 · Figures
import matplotlib.pyplot as plt
FIG = f"{CFG['OUT']}/figures"
a = res[(res.subset == 'all') & (res.condition != 'clean')].set_index('condition')
M = a[['delta_eer', 'delta_fa_bona']].copy(); M['delta_miss_spoof'] = a.miss_spoof_at_thr - cl.loc['all', 'miss_spoof_at_thr']
fig, ax = plt.subplots(figsize=(7, 0.45 * len(M) + 1.5))
v = max(5.0, float(np.nanmax(np.abs(100 * M.values))))
im = ax.imshow(100 * M.values, cmap='RdBu_r', aspect='auto', vmin=-v, vmax=v)
ax.set_xticks(range(3), ['EER rise\n(ranking)', 'real callers rejected\nrise (threshold)', 'fakes accepted\nrise'])
ax.set_yticks(range(len(M)), M.index)
for i in range(M.shape[0]):
    for j in range(3):
        ax.text(j, i, f'{100*M.values[i,j]:+.1f}', ha='center', va='center', fontsize=9)
ax.set_title('SSL-AASIST: change vs clean, percentage points'); fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout(); fig.savefig(f'{FIG}/heatmap_ssl_aasist.png', dpi=200); plt.show()

conds = [c for c in scores if c != 'clean']
fig, axes = plt.subplots(int(np.ceil(len(conds) / 3)), 3, figsize=(13, 3 * np.ceil(len(conds) / 3)), squeeze=False)
for axx, cname in zip(axes.flat, conds):
    d = scores[cname]
    for lab, col, name in ((1, 'tab:green', 'bona fide'), (0, 'tab:red', 'spoof')):
        axx.hist(scores['clean'][scores['clean'].label == lab].score, 50, alpha=.25, color=col, label=f'{name} clean')
        axx.hist(d[d.label == lab].score, 50, histtype='step', color=col, lw=1.5, label=f'{name} {cname}')
    axx.axvline(THR, color='k', ls='--', lw=1); axx.set_title(cname, fontsize=10)
axes.flat[0].legend(fontsize=7)
for axx in list(axes.flat)[len(conds):]:
    axx.axis('off')
fig.suptitle('Score distributions: clean (filled) vs call stage (outline); dashed = transferred threshold')
fig.tight_layout(); fig.savefig(f'{FIG}/score_shift_ssl_aasist.png', dpi=200); plt.show()

In [ ]:
#@title 11 · Run log (keep this with the results)
import platform, transformers
log = dict(when=time.strftime('%Y-%m-%d %H:%M:%S'), cfg=CFG, repo_commit=REPO_COMMIT, checkpoint=os.path.basename(CKPT),
           load_report=load_report, ffmpeg=subprocess.run([FFMPEG, '-version'], capture_output=True, text=True).stdout.split('\n')[0],
           encoders=sorted(enc), conditions={k: v[0] for k, v in CONDS.items()}, torch=torch.__version__,
           transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
           dev_eer=eer_dev, threshold=THR, eval_clean_eer=eer_eval, reproduced=bool(REPRODUCED))
json.dump(log, open(f"{CFG['OUT']}/run_log.json", 'w'), indent=1)
print(json.dumps(log, indent=1))
print('\nSaved in', CFG['OUT'], ': subset.csv, scores/, pilot_results.csv, figures/, run_log.json')

## Added in v1.1
- **Step 12** recomputes "real callers rejected" and "fakes accepted" at three thresholds set on clean dev audio: the dev EER point (used above), the centre of the clean gap, and the point accepting 1% of fakes. When clean EER is 0%, the EER threshold can sit anywhere in the gap, so this checks the finding doesn't depend on where it landed.
- **Step 13** scores the second released checkpoint into `fyp_pilot/run_LA_model`. Steps 1–11 select the first `.pth` alphabetically, which is `Best_LA_model_for_DF.pth` (RawBoost tuned for DF). Step 13 runs `LA_model.pth` (RawBoost tuned for LA, the 0.82% model) so you can compare the two training settings.


In [ ]:
#@title 12 · Does the conclusion depend on the threshold? (uses saved scores, no GPU needed)
import glob, pandas as pd, numpy as np
SC0 = f"{CFG['OUT']}/scores"
dv = pd.read_csv(f'{SC0}/dev__clean.csv')                      # dev, clean: sets every threshold
db, ds = dv[dv.label == 1].score.values, dv[dv.label == 0].score.values
TH = {'dev_EER': P.eer(db, ds)[1],                              # what steps 8-9 used
      'mid_gap': (np.percentile(db, 1) + np.percentile(ds, 99)) / 2,   # centre of the clean gap
      'fakes_1pct': np.percentile(ds, 99)}                      # accept at most 1% of dev fakes
print('thresholds:', {k: round(float(v), 3) for k, v in TH.items()})
rows = []
for f in sorted(glob.glob(f'{SC0}/eval__*.csv')):
    d = pd.read_csv(f); cond = f.split('eval__')[1][:-4]
    b, s = d[d.label == 1].score.values, d[d.label == 0].score.values
    for name, t in TH.items():
        fa, miss = P.rates_at(b, s, t)
        rows.append(dict(condition=cond, threshold=name, real_rejected=fa, fakes_accepted=miss))
R = pd.DataFrame(rows).pivot(index='condition', columns='threshold', values=['real_rejected', 'fakes_accepted'])
order = ['clean', 'clean_rms', 'bandpass_300_3400', 'nb_resample', 'g711u', 'g711u_loss10', 'gsm_fr', 'amrnb_12k2', 'amrnb_4k75', 'opus_nb_8k']
R = R.reindex([c for c in order if c in R.index])
R.to_csv(f"{CFG['OUT']}/threshold_robustness.csv")
display((100 * R).round(1))
print('If "real_rejected" stays high for the same stages under all three thresholds, the finding does not depend on the threshold choice.')


In [ ]:
#@title 13 · Score the other released checkpoint (LA RawBoost setting, the 0.82% model)
# Needs the variables from steps 1-7 (rerun them first if the session restarted; scoring resumes from Drive).
import glob
cks = sorted(glob.glob(f"{CFG['OUT']}/checkpoints/**/*.pth", recursive=True))
print('checkpoints found:', [os.path.basename(p) for p in cks])
CKPT = [p for p in cks if 'LA' in os.path.basename(p) and 'DF' not in os.path.basename(p)][0]
print('now scoring:', os.path.basename(CKPT))
BASE_OUT = CFG['OUT'] if not CFG['OUT'].endswith('/run_LA_model') else CFG['OUT'].rsplit('/run_LA_model', 1)[0]
CFG['OUT'] = f'{BASE_OUT}/run_LA_model'                       # separate folder: nothing from the first run is overwritten
SC = f"{CFG['OUT']}/scores"
for d_ in (SC, f"{CFG['OUT']}/figures"):
    os.makedirs(d_, exist_ok=True)
del model; torch.cuda.empty_cache()
model, load_report = P.build_ssl_aasist(REPO, CKPT, dev)
print('loaded:', load_report)
dev_clean = score_set(subset[subset.part == 'dev'], 'clean')
scores = {c: score_set(EV, c) for c in CONDS}
print('\nDone. Now rerun steps 8, 9, 10, 11 and 12: their outputs go to', CFG['OUT'])


## How to read the results

- **`kind = ranking_breaks`**: EER rises by 10 points or more. The detector can no longer separate real from fake. The mechanism-and-fix paper is possible for this stage.
- **`kind = threshold_only`**: EER barely moves, but many more real callers are rejected at the clean threshold. Ranking survives and only the operating point moves. That points to the threshold paper.
- **`kind = no_effect`**: this stage doesn't break the detector. Also a result: report it.
- **Per-class shift**: if the bona fide scores drop and the spoof scores don't, that's genuine-speech drift. If both drop by the same amount, that's a uniform shift, which simple recalibration may fix.
- `subset = clean_correct` repeats everything on utterances the detector got right when clean. Report both sets.

**Send Maninda:** the reproduction line from step 8, the table from step 9 and the heatmap. Next week adds XLSR-SLS and AASIST, noise suppression and gain control, RTCFake's English real-speech pairs and ADD-C pairs.